# LSTM-arithmetic

## Dataset
- [Arithmetic dataset](https://drive.google.com/file/d/1cMuL3hF9jefka9RyF4gEBIGGeFGZYHE-/view?usp=sharing)

In [ ]:
# ! pip install seaborn
# ! pip install opencc
# ! pip install -U scikit-learn

import numpy as np
import pandas as pd
import torch
import torch.nn
import torch.nn.utils.rnn
import torch.utils.data
import matplotlib.pyplot as plt
import seaborn as sns
import opencc
import os
from sklearn.model_selection import train_test_split

data_path = './data'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 141.6 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [2]:
df_train = pd.read_csv(os.path.join(data_path, 'arithmetic_train.csv')) # df_train: columns: ['src', 'tgt']
df_eval = pd.read_csv(os.path.join(data_path, 'arithmetic_eval.csv')) # df_eval: columns: ['src', 'tgt']
df_train.head()

,src,tgt
0,14*(43+20)=,882
1,(6+1)*5=,35
2,13+32+29=,74
3,31*(3-11)=,-248
4,24*49+1=,1177


In [3]:
# transform the input data to string
df_train['tgt'] = df_train['tgt'].apply(lambda x: str(x)) # transform target to string
df_train['src'] = df_train['src'].add(df_train['tgt']) # concatenate source and target
df_train['len'] = df_train['src'].apply(lambda x: len(x)) # get length of each sequence

df_eval['tgt'] = df_eval['tgt'].apply(lambda x: str(x)) # transform target to string

In [4]:
df_train.head()

,src,tgt,len
0,14*(43+20)=882,882,14
1,(6+1)*5=35,35,10
2,13+32+29=74,74,11
3,31*(3-11)=-248,-248,14
4,24*49+1=1177,1177,12


In [5]:
df_eval.head()

,src,tgt
0,48+43+34=,125
1,30-(48+13)=,-31
2,(21*31)+10=,661
3,2-27-10=,-35
4,(15*20)+24=,324


# Build Dictionary
 - The model cannot perform calculations directly with plain text.
 - Convert all text (numbers/symbols) into numerical representations.
 - Special tokens
    - '&lt;pad&gt;'
        - Each sentence within a batch may have different lengths.
        - The length is padded with '&lt;pad&gt;' to match the longest sentence in the batch.
    - '&lt;eos&gt;'
        - Specifies the end of the generated sequence.
        - Without '&lt;eos&gt;', the model will not know when to stop generating.

In [6]:
char_to_id = {}
id_to_char = {}

# write your code here
# Build a dictionary and give every token in the train dataset an id
# The dictionary should contain <eos> and <pad>
# char_to_id is to conver charactors to ids, while id_to_char is the opposite

# 1. Define the special tokens you want to add to the vocabulary.
special_tokens = ['<pad>', '<eos>']
# 2. Find all unique characters in your training data.
unique_data_chars = sorted(list(set("".join(df_train['src']))))
print('Unique characters in training data:', unique_data_chars)
# 3. Create the full vocabulary list by adding the special tokens at the beginning.
full_vocab = special_tokens + unique_data_chars
print('Full vocabulary list:', full_vocab)
# 4. Build the mapping dictionaries using the full vocabulary list.
for idx, char in enumerate(full_vocab):
    char_to_id[char] = idx
    id_to_char[idx] = char

vocab_size = len(char_to_id)
print('Vocab size{}'.format(vocab_size))
print('Character to ID mapping:', char_to_id)
print('ID to Character mapping:', id_to_char)

Unique characters in training data: ['(', ')', '*', '+', '-', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=']
Full vocabulary list: ['<pad>', '<eos>', '(', ')', '*', '+', '-', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=']
Vocab size18
Character to ID mapping: {'<pad>': 0, '<eos>': 1, '(': 2, ')': 3, '*': 4, '+': 5, '-': 6, '0': 7, '1': 8, '2': 9, '3': 10, '4': 11, '5': 12, '6': 13, '7': 14, '8': 15, '9': 16, '=': 17}
ID to Character mapping: {0: '<pad>', 1: '<eos>', 2: '(', 3: ')', 4: '*', 5: '+', 6: '-', 7: '0', 8: '1', 9: '2', 10: '3', 11: '4', 12: '5', 13: '6', 14: '7', 15: '8', 16: '9', 17: '='}


# Data Preprocessing
 - The data is processed into the format required for the model's input and output. (End with \<eos\> token)


In [7]:
# 1. Create 'char_id_list': This will be the model's INPUT (X).
#    It's the full string "prompt + answer", WITHOUT <eos>.
df_train['char_id_list'] = df_train['src'].apply(
    lambda s: [char_to_id[c] for c in s]
)

# 2. Create 'label_id_list': This will be the model's TARGET (Y).
#    It's the "next token" prediction, masked for the prompt.
df_train['label_id_list'] = df_train.apply(
    lambda row:
        # Pad for the prompt, but one *less* token
        [char_to_id['<pad>']] * (len(row['src']) - len(row['tgt']) - 1) +
        # The target starts with the *first char* of the answer
        [char_to_id[c] for c in row['tgt']] +
        # End with <eos>
        [char_to_id['<eos>']],
    axis=1
)

df_train.head()

,src,tgt,len,char_id_list,label_id_list
0,14*(43+20)=882,882,14,"[8, 11, 4, 2, 11, 10, 5, 9, 7, 3, 17, 15, 15, 9]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 15, 9, 1]"
1,(6+1)*5=35,35,10,"[2, 13, 5, 8, 3, 4, 12, 17, 10, 12]","[0, 0, 0, 0, 0, 0, 0, 10, 12, 1]"
2,13+32+29=74,74,11,"[8, 10, 5, 10, 9, 5, 9, 16, 17, 14, 11]","[0, 0, 0, 0, 0, 0, 0, 0, 14, 11, 1]"
3,31*(3-11)=-248,-248,14,"[10, 8, 4, 2, 10, 6, 8, 8, 3, 17, 6, 9, 11, 15]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 9, 11, 15, 1]"
4,24*49+1=1177,1177,12,"[9, 11, 4, 11, 16, 5, 8, 17, 8, 8, 14, 14]","[0, 0, 0, 0, 0, 0, 0, 8, 8, 14, 14, 1]"


# Hyper Parameters

|Hyperparameter|Meaning|Value|
|-|-|-|
|`batch_size`|Number of data samples in a single batch|64|
|`epochs`|Total number of epochs to train|10|
|`embed_dim`|Dimension of the word embeddings|256|
|`hidden_dim`|Dimension of the hidden state in each timestep of the LSTM|256|
|`lr`|Learning Rate|0.001|
|`grad_clip`|To prevent gradient explosion in RNNs, restrict the gradient range|1|

In [8]:
batch_size = 512 # original 64
epochs = 16 # original 2
embed_dim = 256
hidden_dim = 512 # original 256
lr = 0.001
grad_clip = 1

# Data Batching
- Use `torch.utils.data.Dataset` to create a data generation tool called  `dataset`.
- The, use `torch.utils.data.DataLoader` to randomly sample from the `dataset` and group the samples into batches.

- Example: 1+2-3=0
    - Model input: 1 + 2 - 3 = 0
    - Model output: / / / / / 0 &lt;eos&gt;  (the '/' can be replaced with &lt;pad&gt;)
    - The key for the model's output is that the model does not need to predict the next character of the previous part. What matters is that once the model sees '=', it should start generating the answer, which is '0'. After generating the answer, it should also generate&lt;eos&gt;

In [9]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        # return the amount of data
        return self.sequences.shape[0]

    def __getitem__(self, index):
        # Extract the input data x and the ground truth y from the data
        sequence_row = self.sequences.iloc[index]
        x = sequence_row['char_id_list']
        y = sequence_row['label_id_list']
        return x, y

# collate function, used to build dataloader
def collate_fn(batch):
    batch_x = [torch.tensor(data[0]) for data in batch]
    batch_y = [torch.tensor(data[1]) for data in batch]
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x])
    batch_y_lens = torch.LongTensor([len(y) for y in batch_y])

    # Pad the input sequence
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    pad_batch_y = torch.nn.utils.rnn.pad_sequence(batch_y,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    return pad_batch_x, pad_batch_y, batch_x_lens, batch_y_lens

In [10]:
ds_train = Dataset(df_train[['char_id_list', 'label_id_list']])

In [11]:
# Build dataloader of train set and eval set, collate_fn is the collate function
dl_train = torch.utils.data.DataLoader(ds_train,
                                       batch_size=batch_size,
                                       shuffle=True,
                                       collate_fn=collate_fn)

In [12]:
# New Cell for evaluation dataset and collate function
class Dataset_eval(torch.utils.data.Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        # return the amount of data
        return self.dataframe.shape[0]

    def __getitem__(self, index):
        # Extract the source prompt and target answer
        row = self.dataframe.iloc[index]
        src_prompt = row['src'] # e.g., "1+1="
        tgt_answer = row['tgt'] # e.g., "2"

        # Convert prompt string to a list of IDs
        x_ids = [char_to_id[c] for c in src_prompt]

        # Return the IDs, the original prompt, and the original answer
        return x_ids, src_prompt, tgt_answer

# New collate function for evaluation batches
def collate_fn_eval(batch):
    # batch is a list of (x_ids, src_prompt, tgt_answer)

    batch_x_ids = [torch.tensor(data[0]) for data in batch]
    batch_src_str = [data[1] for data in batch]
    batch_tgt_str = [data[2] for data in batch]

    # Get original lengths *before* padding
    batch_x_lens = torch.LongTensor([len(x) for x in batch_x_ids])

    # Pad the input prompts
    pad_batch_x = torch.nn.utils.rnn.pad_sequence(batch_x_ids,
                                                  batch_first=True,
                                                  padding_value=char_to_id['<pad>'])

    # Return padded IDs, lengths, and the original strings for comparison
    return pad_batch_x, batch_x_lens, batch_src_str, batch_tgt_str

In [13]:
# --- Create the new Eval DataLoader ---
ds_eval = Dataset_eval(df_eval)
dl_eval = torch.utils.data.DataLoader(ds_eval,
                                        batch_size=batch_size, # Use your global batch_size
                                        shuffle=False,
                                        collate_fn=collate_fn_eval)

# Model Design

## Execution Flow
1. Convert all characters in the sentence into embeddings.
2. Pass the embeddings through an LSTM sequentially.
3. The output of the LSTM is passed into another LSTM, and additional layers can be added.
4. The output from all time steps of the final LSTM is passed through a Fully Connected layer.
5. The character corresponding to the maximum value across all output dimensions is selected as the next character.

## Loss Function
Since this is a classification task, Cross Entropy is used as the loss function.

## Gradient Update
Adam algorithm is used for gradient updates.

In [14]:
class CharRNN(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super(CharRNN, self).__init__()

        self.embedding = torch.nn.Embedding(num_embeddings=vocab_size,
                                            embedding_dim=embed_dim,
                                            padding_idx=char_to_id['<pad>'])

        self.rnn_layer1 = torch.nn.LSTM(input_size=embed_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)

        self.rnn_layer2 = torch.nn.LSTM(input_size=hidden_dim,
                                        hidden_size=hidden_dim,
                                        batch_first=True)

        self.linear = torch.nn.Sequential(torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=hidden_dim),
                                          torch.nn.ReLU(),
                                          torch.nn.Linear(in_features=hidden_dim,
                                                          out_features=vocab_size))

    def forward(self, batch_x, batch_x_lens):
        return self.encoder(batch_x, batch_x_lens)

    # The forward pass of the model
    def encoder(self, batch_x, batch_x_lens):
        batch_x = self.embedding(batch_x)

        batch_x = torch.nn.utils.rnn.pack_padded_sequence(batch_x,
                                                          batch_x_lens,
                                                          batch_first=True,
                                                          enforce_sorted=False)

        batch_x, _ = self.rnn_layer1(batch_x)
        batch_x, _ = self.rnn_layer2(batch_x)

        batch_x, _ = torch.nn.utils.rnn.pad_packed_sequence(batch_x,
                                                            batch_first=True)

        batch_x = self.linear(batch_x)

        return batch_x

    def generator(self, start_char, max_len=200):
        """
        Generates a sequence token-by-token, caching the LSTM hidden state
        for efficient, one-step-at-a-time decoding.
        """
        # 1. Setup
        with torch.no_grad():  # Disable gradient calculation for inference
            device = next(self.parameters()).device
            char_list = [char_to_id[c] for c in start_char]

            # ---
            # 2. ENCODER STEP: Process the entire prompt string ("1+1=") once.
            # ---

            # Convert prompt list to a tensor: [1, prompt_length]
            x = torch.LongTensor([char_list]).to(device)

            # Pass the entire prompt through the model
            x_embed = self.embedding(x)

            # Get the output sequence AND the final hidden states (h, c)
            # h1, c1 are the states for rnn_layer1
            # h2, c2 are the states for rnn_layer2
            out1, (h1, c1) = self.rnn_layer1(x_embed)
            out2, (h2, c2) = self.rnn_layer2(out1)

            # Get the logits for the *very last* token of the prompt
            y = self.linear(out2[:, -1, :]) # y shape: [1, vocab_size]

            # Get the first predicted character ID (the one after '=')
            next_char_id = torch.argmax(y, dim=1).item()

            # ---
            # 3. DECODER STEP: Generate new tokens one by one.
            # ---

            # Loop until max_len or <eos>
            while len(char_list) < max_len:
                # Stop if the *next* token is <eos>
                if next_char_id == char_to_id['<eos>']:
                    break

                # Add the valid predicted token to our list
                char_list.append(next_char_id)

                # Prepare the *single* last token as input
                # Input tensor shape: [1, 1] (batch_size=1, seq_len=1)
                x = torch.LongTensor([[next_char_id]]).to(device)

                # Feed the new token AND the cached hidden states

                # 1. Pass (token, h1, c1) into LSTM 1
                x_embed = self.embedding(x) # x_embed shape [1, 1, embed_dim]
                out1, (h1, c1) = self.rnn_layer1(x_embed, (h1, c1)) # (h1, c1) are updated

                # 2. Pass (output of LSTM 1, h2, c2) into LSTM 2
                out2, (h2, c2) = self.rnn_layer2(out1, (h2, c2)) # (h2, c2) are updated

                # Get the output logits for this *single* time step
                y = self.linear(out2[:, -1, :]) # y shape: [1, vocab_size]

                # Get the next predicted character
                next_char_id = torch.argmax(y, dim=1).item()

            # 4. Return
            # After the loop finishes (by max_len or <eos>), convert IDs to chars
            return [id_to_char[ch_id] for ch_id in char_list]

    # New method for batch generation
    def generator_batch(self, batch_x, batch_x_lens, max_len=50):
        """
        Generates sequences for an entire batch, using cached hidden states.
        """
        with torch.no_grad():
            device = next(self.parameters()).device
            batch_size = batch_x.shape[0]

            # --- 1. ENCODER STEP ---
            # Process the entire batch of prompts to get the
            # initial hidden states and the first prediction.

            x_embed = self.embedding(batch_x)

            # Pack the padded batch
            packed_x = torch.nn.utils.rnn.pack_padded_sequence(
                x_embed,
                batch_x_lens.cpu(), # Must be on CPU
                batch_first=True,
                enforce_sorted=False
            )

            # Get final outputs and hidden states from both layers
            out1, (h1, c1) = self.rnn_layer1(packed_x)
            out2, (h2, c2) = self.rnn_layer2(out1)

            # Unpack to get the full sequence of logits
            unpacked_out, _ = torch.nn.utils.rnn.pad_packed_sequence(
                out2,
                batch_first=True
            )
            all_logits = self.linear(unpacked_out)

            # Get the logits for the *last actual token* of each prompt
            last_indices = batch_x_lens.to(device) - 1
            last_logits = all_logits[
                torch.arange(batch_size, device=device),
                last_indices
            ] # Shape: [batch_size, vocab_size]

            # Get the first predicted token for each item in the batch
            next_token_ids = torch.argmax(last_logits, dim=1) # Shape: [batch_size]

            # --- 2. DECODER STEP ---
            # Generate tokens one-by-one, in parallel for the whole batch.

            # Tensor to store all generated token IDs
            all_generated_ids_tensor = torch.full(
                (batch_size, max_len),
                char_to_id['<pad>'],
                dtype=torch.long,
                device=device
            )

            # Boolean tensor to track which sequences are "done"
            done_tracking = torch.zeros(batch_size, dtype=torch.bool, device=device)

            # (h1, c1, h2, c2) are now the initial decoder states

            for i in range(max_len):
                # Store the tokens from the *previous* step
                # Only store if the sequence is NOT already done
                all_generated_ids_tensor[~done_tracking, i] = next_token_ids[~done_tracking]

                # Check which sequences just generated <eos>
                newly_done = (next_token_ids == char_to_id['<eos>'])
                done_tracking.logical_or_(newly_done)

                # If all sequences in the batch are done, stop early
                if done_tracking.all():
                    break

                # Prepare input for the *next* step
                # Use <pad> as input for sequences that are already done
                input_tokens = next_token_ids

                # Feed the single token (per batch item) into the LSTMs
                x = input_tokens.unsqueeze(1) # Shape: [batch_size, 1]
                x_embed = self.embedding(x)  # Shape: [batch_size, 1, embed_dim]

                # Run one step of the RNNs using the cached hidden states
                out1, (h1, c1) = self.rnn_layer1(x_embed, (h1, c1))
                out2, (h2, c2) = self.rnn_layer2(out1, (h2, c2))

                # Get the logits for this single time step
                logits = self.linear(out2.squeeze(1)) # Shape: [batch_size, vocab_size]

                # Get the next predicted tokens
                next_token_ids = torch.argmax(logits, dim=1)

            # --- 3. POST-PROCESSING ---
            # Convert the ID tensors back to lists of strings
            final_predictions = []
            for i in range(batch_size):
                pred_string = ""
                for token_id in all_generated_ids_tensor[i].tolist():
                    if token_id == char_to_id['<eos>']:
                        break # Stop at <eos>
                    if token_id == char_to_id['<pad>']:
                        break # Stop at <pad> (end of generation)
                    pred_string += id_to_char[token_id]
                final_predictions.append(pred_string)

            return final_predictions

In [15]:
torch.manual_seed(2)


device = 'cuda'

model = CharRNN(vocab_size,
                embed_dim,
                hidden_dim)

In [16]:
# Cross-entropy loss function. The loss function should ignore <pad>
criterion = torch.nn.CrossEntropyLoss(ignore_index=char_to_id['<pad>'])
# Use Adam or AdamW for Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Training
1. The outer `for` loop controls the `epoch`
    1. The inner `for` loop uses `data_loader` to retrieve batches.
        1. Pass the batch to the `model` for training.
        2. Compare the predicted results `batch_pred_y` with the true labels `batch_y` using Cross Entropy to calculate the loss `loss`
        3. Use `loss.backward` to automatically compute the gradients.
        4. Use `torch.nn.utils.clip_grad_value_` to limit the gradient values between `-grad_clip` &lt; and &lt; `grad_clip`.
        5. Use `optimizer.step()` to update the model (backpropagation).
2.  After every `1000` batches, output the current loss to monitor whether it is converging.

In [17]:
from tqdm import tqdm
from copy import deepcopy

model = model.to(device)
model.train()
i = 0

for epoch in range(1, epochs+1):
    # The process bar
    bar = tqdm(dl_train, desc=f"Train epoch {epoch}")

    for batch_x, batch_y, batch_x_lens, batch_y_lens in bar:
        # Write your code here
        # Clear the gradient
        optimizer.zero_grad(set_to_none=True)

        batch_pred_y = model(batch_x.to(device), batch_x_lens)

        # Write your code here
        # Input the prediction and ground truths to loss function
        # B, T, V = batch_pred_y.shape
        # loss = criterion(
        #     batch_pred_y.reshape(B*T, V),
        #     batch_y.to(device).reshape(B*T)
        # )

        loss = criterion(batch_pred_y.view(-1, vocab_size),
                         batch_y.to(device).view(-1))
        # Back propagation
        loss.backward()

        # gradient clipping
        torch.nn.utils.clip_grad_value_(model.parameters(), grad_clip)

        # Write your code here
        # Optimize parameters in the model
        optimizer.step()

        i+=1
        if i%50==0:
            bar.set_postfix(loss = loss.item())

    # --- Start Evaluation ---
    model.eval()
    matched = 0
    total = 0

    bar_eval = tqdm(dl_eval, desc=f"Validation epoch {epoch}")
    with torch.no_grad():
        # Loop over BATCHES from the DataLoader
        for pad_batch_x, batch_x_lens, batch_src_str, batch_tgt_str in bar_eval:

            # Move prompt tensors to the GPU
            pad_batch_x = pad_batch_x.to(device)
            # batch_x_lens stays on CPU for pack_padded_sequence

            # Run the new batch generator
            # `predictions` will be a list of strings (e.g., ['2', '12', '-5'])
            predictions = model.generator_batch(
                pad_batch_x,
                batch_x_lens,
                max_len=50 # Set a reasonable max answer length
            )

            # Check the batch of predictions against the batch of targets
            for pred, target in zip(predictions, batch_tgt_str):
                if pred == target:
                    matched += 1
                total += 1

            if total > 0:
                bar_eval.set_postfix(EM=matched / total)

    print(f"\nValidation EM: {matched / total}")
    model.train()  # back to train mode for next epoch

Validation epoch 1: 100%|██████████| 515/515 [00:29<00:00, 17.38it/s, EM=0.6]



Validation EM: 0.6000341880341881


Validation epoch 2: 100%|██████████| 515/515 [00:28<00:00, 18.12it/s, EM=0.732]



Validation EM: 0.7323532763532764


Validation epoch 3: 100%|██████████| 515/515 [00:28<00:00, 18.02it/s, EM=0.85]


...


Validation EM: 0.9641025641025641


Validation epoch 15: 100%|██████████| 515/515 [00:29<00:00, 17.38it/s, EM=0.971]



Validation EM: 0.9706780626780627


Validation epoch 16: 100%|██████████| 515/515 [00:29<00:00, 17.57it/s, EM=0.963]


Validation EM: 0.963255460588794
